In [2]:
unembedding_matrix.shape

(50257, 768)

In [35]:
hash_keys

array([[-1, -1,  1, ..., -1,  1,  1],
       [-1, -1, -1, ...,  1, -1,  1],
       [-1, -1, -1, ...,  1, -1, -1],
       ...,
       [-1,  1, -1, ...,  1,  1,  1],
       [ 1, -1,  1, ..., -1,  1,  1],
       [ 1, -1, -1, ...,  1, -1, -1]], dtype=int32)

In [54]:
len(buckets)


1023

In [58]:
import torch
import numpy as np

def lsh_partition(vectors, num_hashes=10, hash_size=32, seed=42):
    """
    Partition vectors into buckets using Locality Sensitive Hashing (LSH).
    Args:
        vectors (torch.Tensor): Input vectors (n_vectors x dim).
        num_hashes (int): Number of hash functions.
        hash_size (int): Length of each hash vector.
        seed (int): Random seed for reproducibility.
    Returns:
        buckets (dict): A dictionary where keys are hash values, and values are lists of vector indices.
    """
    torch.manual_seed(seed)
    random_planes = torch.randn((num_hashes, vectors.size(1), hash_size), device=vectors.device)
    projections = (vectors @ random_planes) # Binary hashing
    hash_keys = projections.sum(dim=2).sign().transpose(0,1)
    hash_keys = hash_keys.cpu().numpy().astype(np.int32)

    # Group vectors into buckets
    buckets = {}
    for idx, key in enumerate(hash_keys):
        key_tuple = tuple(key)
        if key_tuple not in buckets:
            buckets[key_tuple] = []
        buckets[key_tuple].append(idx)
    return buckets, hash_keys

def compute_sparse_distances_from_buckets(vectors, buckets, threshold_quantile=0.98):
    """
    Compute sparse distance matrix by processing buckets.
    Args:
        vectors (torch.Tensor): Input vectors (n_vectors x dim).
        buckets (dict): LSH buckets.
        threshold (float): Distance threshold for sparsity.
    Returns:
        sparse_matrix (torch.sparse.Tensor): Sparse distance matrix.
    """
    indices = []
    values = []

    for bucket_indices in buckets.values():
        if len(bucket_indices) < 2:
            continue  # Skip buckets with fewer than 2 vectors

        # Extract vectors in this bucket
        bucket_vectors = vectors[bucket_indices]
        
        # Compute pairwise distances within the bucket
        distances = torch.cdist(bucket_vectors, bucket_vectors, p=2)
        # print(distances.shape)
        # print(torch.median(distances))
        threshold = torch.quantile(distances, threshold_quantile)
        mask = distances < threshold
        row_indices, col_indices = torch.where(mask)

        # Convert to global indices
        global_row_indices = torch.tensor(bucket_indices, device=vectors.device)[row_indices]
        global_col_indices = torch.tensor(bucket_indices, device=vectors.device)[col_indices]

        # Append results
        indices.append(torch.stack([global_row_indices, global_col_indices], dim=0))
        values.append(distances[mask])

    # Combine results into a sparse matrix
    if indices:
        indices = torch.cat(indices, dim=1)
        values = torch.cat(values)
    else:
        indices = torch.empty((2, 0), dtype=torch.int64, device=vectors.device)
        values = torch.empty(0, device=vectors.device)

    values = values.to(device='cpu')
    indices = indices.to(device='cpu')
    sparse_matrix = torch.sparse_coo_tensor(
        indices=indices,
        values=values,
        size=(vectors.size(0), vectors.size(0)),
    )
    return sparse_matrix

# Example usage
n_vectors = 10000
dim = 100
vectors = torch.randn(n_vectors, dim, device='mps')  # Random vectors on MPS

# Step 1: Partition vectors using LSH
num_hashes = 10
hash_size = 84
buckets, hash_keys = lsh_partition(vectors, num_hashes=num_hashes, hash_size=hash_size)

# Step 2: Compute sparse distance matrix within buckets
threshold_quantile = 0.995
sparse_matrix = compute_sparse_distances_from_buckets(vectors, buckets, threshold_quantile=threshold_quantile)

# Inspect sparse matrix
print(sparse_matrix)

tensor(indices=tensor([[   0,    0,    0,  ..., 8945, 8945, 8945],
                       [   0,  939, 1091,  ..., 7815, 8694, 8945]]),
       values=tensor([ 0.0000, 12.3384, 12.9716,  ..., 12.9356, 13.6662,
                       0.0000]),
       size=(10000, 10000), nnz=123095, layout=torch.sparse_coo)


In [50]:
hash_keys.shape

(10, 10000)

In [38]:
for bucket_indices in buckets.values():
    if len(bucket_indices) > 1:
        print(bucket_indices)

In [60]:
import torch
from transformers import GPT2LMHeadModel
import networkx as nx
import numpy as np
from scipy.spatial.distance import cosine

# Load GPT2 model
model = GPT2LMHeadModel.from_pretrained('gpt2')

# Get unembedding matrix (word embeddings)
unembedding_matrix = model.lm_head.weight.detach()

In [91]:
from transformers import GPT2Tokenizer

# Initialize the tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Filter token positions to exclude non-latin, non-numeric, and special characters
def filter_token_positions():
    # Get the vocabulary from the tokenizer
    vocab = tokenizer.get_vocab()
    
    # Define a function to check if a token is valid
    def is_valid_token(token):
        return all(c.isalpha() and c in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ" or c in "!@#$%^&*()[]{}/\\|<>;:,. " for c in token)

    # Filter the token positions
    valid_token_indices = torch.tensor([i for token,i in vocab.items() if is_valid_token(token)])
    
    return valid_token_indices

# Get filtered token positions
filtered_token_positions = filter_token_positions()

/Users/solar/miniconda3/envs/pytorch/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [96]:
vocab = tokenizer.get_vocab()
filtered_vocab = {token: i for token, i in vocab.items() if i in filtered_token_positions}
# sort by length 
filtered_vocab = sorted(filtered_vocab.items(), key=lambda x: len(x[0]), reverse=True)

In [97]:
unembedding_matrix = unembedding_matrix[filtered_token_positions]

num_hashes = 10
hash_size = 84
buckets, hash_keys = lsh_partition(unembedding_matrix, num_hashes=num_hashes, hash_size=hash_size)

# Step 2: Compute sparse distance matrix within buckets
threshold_quantile = 0.995
sparse_similarity_matrix = compute_sparse_distances_from_buckets(unembedding_matrix, buckets, threshold_quantile=threshold_quantile)

print('percentage of non-zero connections:', sparse_similarity_matrix.coalesce().indices().shape[1]/unembedding_matrix.shape[0]**2)

percentage of non-zero connections: 0.00419918123936229


In [98]:
sparse_similarity_matrix

tensor(indices=tensor([[    1,     1,     1,  ..., 14922, 13915, 15048],
                       [    1,    45,   764,  ..., 14922, 13915, 15048]]),
       values=tensor([3.3829e-03, 3.9985e+00, 4.4493e+00,  ...,
                      0.0000e+00, 0.0000e+00, 0.0000e+00]),
       size=(15082, 15082), nnz=955174, layout=torch.sparse_coo)

In [104]:
sparse_similarity_matrix.coalesce().indices().to('cpu').numpy().T[0]

array([1, 1])

In [111]:
# create graph
G = nx.Graph()
G.add_nodes_from(range(unembedding_matrix.shape[0]))
indices = sparse_similarity_matrix.coalesce().indices().to('cpu').numpy().T
values = sparse_similarity_matrix.coalesce().values().to('cpu').numpy().T
for i,j,w in zip(indices[:,0], indices[:,1], values):
    G.add_edge(i,j, weight=w)

In [112]:
len(G.edges())

485062

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.00338291, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.00195312,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], dtype=float32)

In [119]:
from geomechinterp.curvature.balanced_forman_curvature import balanced_forman_curvature

# get adjacency matrix
A = nx.adjacency_matrix(G)
A = A.todense()
A = (A > 0).astype(int)
C = balanced_forman_curvature(A)

In [123]:
C

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.00338291, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.00195312,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], dtype=float32)

In [121]:
C.max()

0.0